# Meteorological Dataset - Data Processing

Processing pipeline for hourly weather data in Madrid, sourced from the [Meteostat](https://meteostat.net) API. The final dataset covers the same time range as the traffic dataset (2023-01-01 00:00 to 2026-02-28 23:00 = 27,720 hours).

## Specification

1. **Fetch primary station data**: Retrieve hourly weather observations from **Madrid Retiro** (Meteostat station `08222`) for the full time range. Columns: `temp` (°C), `rhum` (% relative humidity), `prcp` (mm precipitation), `snwd` (snow depth), `wdir` (° wind direction), `wspd` (km/h wind speed), `wpgt` (km/h wind gust), `pres` (hPa pressure), `tsun` (min sunshine), `cldc` (oktas cloud cover), `coco` (weather condition code). Note: `snwd`, `wpgt`, and `tsun` are entirely missing for this station.
2. **Inspect data quality**: Examine dtypes, summary statistics, null counts, and overall shape. The primary station returns 27,330 of 27,720 expected hours (390 missing).
3. **Reindex to complete hourly range**: Reindex to the full 27,720-hour timeline so missing timestamps become explicit NaN rows.
4. **Patch with secondary station — Cuatro Vientos** (`08223`): Fetch hourly data from Madrid Cuatro Vientos, reindex to the same timeline, and use `combine_first()` to fill NaN rows in the primary dataset. Reduces missing `temp` values from 390 to 26.
5. **Patch with tertiary station — Barajas** (`08221`): Fetch hourly data from Madrid Barajas airport, reindex, and apply a second `combine_first()` to fill the remaining 26 hours where both Retiro and Cuatro Vientos were offline. Reduces missing `temp` values to 0.
6. **Impute missing `prcp`**: After station patching, 414 hours still have missing precipitation. The `coco` (weather condition code) column — which has zero nulls — is used to decide the imputation strategy:
   - **Dry conditions** (coco 1–4: clear, fair, cloudy, overcast) — 337 hours → impute `prcp = 0.0`. In the observed data, 100% of coco 1–3 hours have zero precipitation; coco 4 is near-zero (mean 0.14 mm).
   - **Rain conditions** (coco 7–8: light rain, rain) — 77 hours → impute with the **median `prcp`** for the corresponding coco code, computed from hours where `prcp` is known. This preserves the typical precipitation intensity for each condition.
7. **Export**: Write the fully patched dataset (`df_fully_patched`) to `dataset/processed/meteosat.csv`.

---

> **Note — Station patching priority**
>
> Data is sourced with a priority chain: **Retiro → Cuatro Vientos → Barajas**. Retiro is the primary station as it is closest to central Madrid. The 390 hours missing from Retiro were concentrated in two periods, and were fully recovered using the two fallback stations. After patching, `temp`, `rhum`, `wdir`, `wspd`, `cldc`, and `coco` have zero missing values. `prcp` had 414 remaining nulls, resolved via coco-based imputation (step 6). `snwd`, `wpgt`, and `tsun` remain mostly or entirely missing across all three stations.

In [119]:
%pip install pandas numpy pyarrow meteostat
import pandas as pd
import numpy as np
from pathlib import Path
import glob

ERROR:root:code for hash blake2b was not found.
Traceback (most recent call last):
  File "/Users/jai/.pyenv/versions/3.12.1/lib/python3.12/hashlib.py", line 245, in <module>
    globals()[__func_name] = __get_hash(__func_name)
                             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jai/.pyenv/versions/3.12.1/lib/python3.12/hashlib.py", line 129, in __get_openssl_constructor
    return __get_builtin_constructor(name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jai/.pyenv/versions/3.12.1/lib/python3.12/hashlib.py", line 123, in __get_builtin_constructor
    raise ValueError('unsupported hash type ' + name)
ValueError: unsupported hash type blake2b
ERROR:root:code for hash blake2s was not found.
Traceback (most recent call last):
  File "/Users/jai/.pyenv/versions/3.12.1/lib/python3.12/hashlib.py", line 245, in <module>
    globals()[__func_name] = __get_hash(__func_name)
                             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jai/.pyenv/versions/3.1

In [120]:
from datetime import datetime
import meteostat as ms

# Set time period
start = datetime(2023, 1, 1)
end = datetime(2026, 2, 28, 23, 59)

# Get hourly data
ts = ms.hourly(ms.Station(id='08222'), start, end)
df = ts.fetch()

# Print DataFrame
df

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,6.1,80,0.0,<NA>,54,3.6,<NA>,1027.0,<NA>,1,1
2023-01-01 01:00:00,6.2,78,0.0,<NA>,90,4.0,<NA>,1026.6,<NA>,2,1
2023-01-01 02:00:00,5.8,78,0.0,<NA>,66,5.0,<NA>,1026.5,<NA>,1,2
2023-01-01 03:00:00,4.0,86,0.0,<NA>,32,5.0,<NA>,1026.4,<NA>,2,2
2023-01-01 04:00:00,3.8,83,0.0,<NA>,31,5.8,<NA>,1025.8,<NA>,1,2
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-28 19:00:00,12.1,60,0.0,<NA>,47,11.9,<NA>,1022.8,<NA>,1,2
2026-02-28 20:00:00,10.8,64,0.0,<NA>,42,11.9,<NA>,1023.4,<NA>,8,1
2026-02-28 21:00:00,10.0,66,0.0,<NA>,31,13.0,<NA>,1024.5,<NA>,8,3


In [121]:
df.head()

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,6.1,80,0.0,<NA>,54,3.6,<NA>,1027.0,<NA>,1,1
2023-01-01 01:00:00,6.2,78,0.0,<NA>,90,4.0,<NA>,1026.6,<NA>,2,1
2023-01-01 02:00:00,5.8,78,0.0,<NA>,66,5.0,<NA>,1026.5,<NA>,1,2
2023-01-01 03:00:00,4.0,86,0.0,<NA>,32,5.0,<NA>,1026.4,<NA>,2,2
2023-01-01 04:00:00,3.8,83,0.0,<NA>,31,5.8,<NA>,1025.8,<NA>,1,2


In [122]:
df.dtypes

temp    Float64
rhum      UInt8
prcp    Float64
snwd     UInt16
wdir     UInt16
wspd    Float64
wpgt    Float64
pres    Float64
tsun      UInt8
cldc      UInt8
coco      UInt8
dtype: object

In [123]:
df.describe()

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
count,27330.0,27330.0,27251.0,0.0,27330.0,27330.0,0.0,27330.0,0.0,27330.0,27251.0
mean,15.32809,60.447859,0.061634,<NA>,157.30333,11.613224,<NA>,1017.687933,<NA>,3.739188,2.652453
std,9.31245,25.982064,0.38279,<NA>,106.27549,5.904646,<NA>,7.13068,<NA>,3.443277,2.379356
min,-4.3,5.0,0.0,<NA>,0.0,0.4,<NA>,988.1,<NA>,0.0,1.0
25%,8.1,38.0,0.0,<NA>,43.0,7.2,<NA>,1013.4,<NA>,0.0,1.0
50%,13.9,63.0,0.0,<NA>,178.0,10.4,<NA>,1017.1,<NA>,3.0,3.0
75%,22.0,84.0,0.0,<NA>,244.0,15.1,<NA>,1021.7,<NA>,8.0,3.0
max,40.3,100.0,12.3,<NA>,360.0,46.4,<NA>,1041.8,<NA>,8.0,19.0


In [124]:
df.shape

(27330, 11)

In [125]:
df.isnull().sum()

temp        0
rhum        0
prcp       79
snwd    27330
wdir        0
wspd        0
wpgt    27330
pres        0
tsun    27330
cldc        0
coco       79
dtype: int64

In [126]:
df.head(1000)

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,6.1,80,0.0,<NA>,54,3.6,<NA>,1027.0,<NA>,1,1
2023-01-01 01:00:00,6.2,78,0.0,<NA>,90,4.0,<NA>,1026.6,<NA>,2,1
2023-01-01 02:00:00,5.8,78,0.0,<NA>,66,5.0,<NA>,1026.5,<NA>,1,2
2023-01-01 03:00:00,4.0,86,0.0,<NA>,32,5.0,<NA>,1026.4,<NA>,2,2
2023-01-01 04:00:00,3.8,83,0.0,<NA>,31,5.8,<NA>,1025.8,<NA>,1,2
...,...,...,...,...,...,...,...,...,...,...,...
2023-02-11 11:00:00,6.2,60,0.0,<NA>,59,13.7,<NA>,1035.5,<NA>,0,1
2023-02-11 12:00:00,8.6,48,0.0,<NA>,114,19.1,<NA>,1034.2,<NA>,5,1
2023-02-11 13:00:00,9.6,42,0.0,<NA>,116,16.6,<NA>,1032.5,<NA>,2,3


In [127]:
df.columns

Index(['temp', 'rhum', 'prcp', 'snwd', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun',
       'cldc', 'coco'],
      dtype='str')

In [128]:
import pandas as pd

# 1. Create a continuous date range of hourly frequencies
expected_dates = pd.date_range(start=start, end=end, freq='h')

# 2. Find exactly which timestamps are missing
missing_hours = expected_dates.difference(df.index)
print(f"Number of missing hours: {len(missing_hours)}")
print("First few missing dates:")
missing_df = pd.DataFrame({'Missing_Hour': missing_hours})
missing_df



Number of missing hours: 390
First few missing dates:


,Missing_Hour
0,2024-08-28 01:00:00
1,2024-08-28 02:00:00
2,2024-08-28 03:00:00
3,2024-08-28 04:00:00
4,2024-08-28 05:00:00
...,...
385,2026-02-13 13:00:00
386,2026-02-13 14:00:00
387,2026-02-13 15:00:00
388,2026-02-13 16:00:00


In [129]:
# 3. Reindex your dataframe so it has all 27,720 rows
# This will insert the missing hours and fill their columns with NaN
df_continuous = df.reindex(expected_dates)

# Verify the new shape
print(f"New shape: {df_continuous.shape}")

New shape: (27720, 11)


In [130]:
from datetime import datetime
import meteostat as ms

# Set time period
start = datetime(2023, 1, 1)
end = datetime(2026, 2, 28, 23, 59)

# Get hourly data
ts2 = ms.hourly(ms.Station(id='08223'), start, end)
df2 = ts2.fetch()

# Print DataFrame
df2

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,7.0,76,0.0,<NA>,20,5.4,<NA>,1026.0,<NA>,1,2
2023-01-01 01:00:00,7.0,76,0.0,<NA>,20,5.4,<NA>,1026.0,<NA>,2,2
2023-01-01 02:00:00,8.0,71,0.0,<NA>,80,3.6,<NA>,1026.0,<NA>,2,2
2023-01-01 03:00:00,8.0,71,0.0,<NA>,66,1.8,<NA>,1025.0,<NA>,3,2
2023-01-01 04:00:00,8.0,71,0.0,<NA>,58,3.6,<NA>,1025.0,<NA>,1,3
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-28 19:00:00,14.0,47,0.0,<NA>,60,1.0,<NA>,1022.0,<NA>,2,2
2026-02-28 20:00:00,14.0,48,0.0,<NA>,50,9.0,<NA>,1023.0,<NA>,2,2
2026-02-28 21:00:00,13.0,51,0.0,<NA>,40,9.0,<NA>,1024.0,<NA>,2,3


In [131]:
# 1. First, make sure df2 (Cuatro Vientos) is also reindexed to the exact same timeline!
# This ensures the timestamps align perfectly before we merge them.
df2_continuous = df2.reindex(expected_dates)

# 2. Patch the holes in Retiro with data from Cuatro Vientos
# combine_first() prioritizes df_continuous. If it sees a NaN, it grabs the value from df2_continuous.
df_patched = df_continuous.combine_first(df2_continuous)

# 3. Verify how many missing values are left (hopefully zero!)
print("Missing values in Retiro BEFORE patching:")
print(df_continuous['temp'].isnull().sum())

print("\nMissing values in Retiro AFTER patching:")
print(df_patched['temp'].isnull().sum())

Missing values in Retiro BEFORE patching:
390

Missing values in Retiro AFTER patching:
26


In [132]:
df

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,6.1,80,0.0,<NA>,54,3.6,<NA>,1027.0,<NA>,1,1
2023-01-01 01:00:00,6.2,78,0.0,<NA>,90,4.0,<NA>,1026.6,<NA>,2,1
2023-01-01 02:00:00,5.8,78,0.0,<NA>,66,5.0,<NA>,1026.5,<NA>,1,2
2023-01-01 03:00:00,4.0,86,0.0,<NA>,32,5.0,<NA>,1026.4,<NA>,2,2
2023-01-01 04:00:00,3.8,83,0.0,<NA>,31,5.8,<NA>,1025.8,<NA>,1,2
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-28 19:00:00,12.1,60,0.0,<NA>,47,11.9,<NA>,1022.8,<NA>,1,2
2026-02-28 20:00:00,10.8,64,0.0,<NA>,42,11.9,<NA>,1023.4,<NA>,8,1
2026-02-28 21:00:00,10.0,66,0.0,<NA>,31,13.0,<NA>,1024.5,<NA>,8,3


In [133]:
# Create a new DataFrame containing ONLY the rows where 'temp' is still NaN
remaining_missing = df_patched[df_patched['temp'].isnull()]

# Display all 26 rows
print("The 26 hours where both stations were offline:")
print(remaining_missing)

# Optional: If you want to see just the timestamps without the clutter of the other columns
print("\nJust the exact dates and times:")
print(remaining_missing.index.tolist())

The 26 hours where both stations were offline:
                     temp  rhum  prcp  snwd  wdir  wspd  wpgt  pres  tsun  \
2025-03-20 21:00:00  <NA>  <NA>  <NA>  <NA>   250  16.6  <NA>  <NA>  <NA>   
2026-02-11 13:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 14:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 15:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 16:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 17:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 19:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 20:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 21:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 22:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-11 23:00:00  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>   
2026-02-12 01:00:00  <NA>  <N

In [134]:
from datetime import datetime
import meteostat as ms

# Set time period
start = datetime(2023, 1, 1)
end = datetime(2026, 2, 28, 23, 59)

# Get hourly data
ts3 = ms.hourly(ms.Station(id='08221'), start, end)
df3 = ts3.fetch()

# Print DataFrame
df3

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
time,,,,,,,,,,,
2023-01-01 00:00:00,4.1,89,0.0,<NA>,330,5.4,<NA>,1028.9,<NA>,2,2
2023-01-01 01:00:00,5.0,87,0.0,<NA>,340,7.6,<NA>,1026.0,<NA>,3,2
2023-01-01 02:00:00,4.0,92,0.0,<NA>,360,3.6,<NA>,1028.2,<NA>,3,2
2023-01-01 03:00:00,4.2,91,0.0,<NA>,0,0.0,<NA>,1027.9,<NA>,6,2
2023-01-01 04:00:00,4.5,94,0.0,<NA>,260,1.8,<NA>,1027.7,<NA>,6,2
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-28 19:00:00,14.0,47,0.0,<NA>,40,3.0,13.0,1022.0,<NA>,2,1
2026-02-28 20:00:00,13.0,54,0.0,<NA>,60,9.0,14.8,1023.0,<NA>,4,2
2026-02-28 21:00:00,12.0,58,0.0,<NA>,60,7.0,16.7,1023.0,<NA>,2,2


In [135]:
# 1. Fetch the actual DataFrame (this was missing!)
df3 = ts3.fetch()

# 2. Reindex the Barajas DataFrame to match your master timeline exactly
df3_continuous = df3.reindex(expected_dates)

# 3. Apply the final patch!
# This looks at df_patched, and wherever it sees a NaN, it pulls from df3_continuous
df_fully_patched = df_patched.combine_first(df3_continuous)

# 4. The moment of truth: Let's check the missing count
print("Missing values BEFORE Barajas:", df_patched['temp'].isnull().sum())
print("Missing values AFTER Barajas:", df_fully_patched['temp'].isnull().sum())

Missing values BEFORE Barajas: 26
Missing values AFTER Barajas: 0


In [136]:
df_fully_patched.describe()

,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
count,27720.0,27720.0,27306.0,1085.0,27720.0,27720.0,20280.0,27720.0,0.0,27720.0,27720.0
mean,15.314787,60.562626,0.061862,0.0,157.259488,11.605732,16.135508,1017.624921,<NA>,3.767027,2.662193
std,9.307272,25.915613,0.382649,0.0,106.317667,5.931192,7.077537,7.126112,<NA>,3.437261,2.37646
min,-4.3,5.0,0.0,0.0,0.0,0.0,0.0,988.1,<NA>,0.0,1.0
25%,8.1,38.0,0.0,0.0,43.0,7.2,11.1,1013.3,<NA>,0.0,1.0
50%,13.9,63.0,0.0,0.0,177.0,10.4,13.0,1017.1,<NA>,3.0,2.0
75%,22.0,84.0,0.0,0.0,244.0,15.1,20.4,1021.6,<NA>,8.0,3.0
max,40.3,100.0,12.3,0.0,360.0,46.4,51.8,1041.8,<NA>,8.0,19.0


In [137]:
df_fully_patched.isnull().sum()

temp        0
rhum        0
prcp      414
snwd    26635
wdir        0
wspd        0
wpgt     7440
pres        0
tsun    27720
cldc        0
coco        0
dtype: int64

## 6. Impute missing `prcp` using weather condition codes

The 414 hours with missing `prcp` all have a valid `coco` (weather condition) value. Cross-referencing with known data:
- **coco 1–3** (clear/fair/cloudy): 100% of observed hours have `prcp = 0` → safe to fill with 0.
- **coco 4** (overcast): near-zero precipitation (mean 0.14 mm) → fill with 0.
- **coco 7** (light rain): all observed hours have `prcp > 0`, median ~0.2 mm → fill with median.
- **coco 8** (rain): 99.6% have `prcp > 0`, median ~0.5 mm → fill with median.

In [138]:
# Compute median prcp per coco code from known data
median_prcp_by_coco = df_fully_patched[df_fully_patched['prcp'].notna()].groupby('coco')['prcp'].median()
print("Median prcp by coco code:")
print(median_prcp_by_coco)

# Identify rows with missing prcp
missing_prcp = df_fully_patched['prcp'].isna()
print(f"\nprcp nulls before imputation: {missing_prcp.sum()}")

# Dry conditions (coco 1-4): fill with 0
dry_mask = missing_prcp & df_fully_patched['coco'].isin([1, 2, 3, 4])
df_fully_patched.loc[dry_mask, 'prcp'] = 0.0
print(f"  Filled {dry_mask.sum()} dry-condition hours with 0.0")

# Rain conditions (coco 7, 8): fill with median prcp for that coco code
for code in [7, 8]:
    rain_mask = missing_prcp & (df_fully_patched['coco'] == code)
    median_val = median_prcp_by_coco[code]
    df_fully_patched.loc[rain_mask, 'prcp'] = median_val
    print(f"  Filled {rain_mask.sum()} coco={code} hours with median {median_val:.1f} mm")

print(f"\nprcp nulls after imputation: {df_fully_patched['prcp'].isna().sum()}")

Median prcp by coco code:
coco
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
7     0.2
8     0.5
9     1.8
12    0.4
15    0.7
17    0.2
18    1.3
19    0.5
Name: prcp, dtype: Float64

prcp nulls before imputation: 414
  Filled 337 dry-condition hours with 0.0
  Filled 57 coco=7 hours with median 0.2 mm
  Filled 20 coco=8 hours with median 0.5 mm

prcp nulls after imputation: 0


In [140]:
df_fully_patched.isnull().sum()

temp        0
rhum        0
prcp        0
snwd    26635
wdir        0
wspd        0
wpgt     7440
pres        0
tsun    27720
cldc        0
coco        0
dtype: int64

In [139]:
from pathlib import Path
Path('dataset/processed').mkdir(parents=True, exist_ok=True)
df_fully_patched.index.name = 'fecha'
df_fully_patched.to_csv('dataset/processed/meteosat.csv', index=True)
print(f'Saved {len(df_fully_patched):,} rows to dataset/processed/meteosat.csv')

Saved 27,720 rows to dataset/processed/meteosat.csv
